In [1]:
pip install cantera

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 4.4 MB/s eta 0:00:00


In [2]:
import cantera as ct
print(ct.__version__)

3.2.0


In [3]:
import cantera as ct

def simulate_flame_temperature(phi, T_init, pressure_atm):
    gas = ct.Solution("gri30.yaml")

    pressure = pressure_atm * ct.one_atm
    gas.TP = T_init, pressure

    # Methane-air combustion
    gas.set_equivalence_ratio(phi, fuel="CH4", oxidizer={"O2": 1.0, "N2": 3.76})

    gas.equilibrate("HP")  # Constant enthalpy & pressure

    flame_temperature = gas.T
    return flame_temperature

In [4]:
simulate_flame_temperature(
    phi=1.0,
    T_init=300,
    pressure_atm=1
)

2225.524583476979

In [5]:
import random
import pandas as pd

data = []
NUM_SAMPLES = 1000

for i in range(NUM_SAMPLES):
    phi = random.uniform(0.6, 1.4)
    T_init = random.uniform(300, 800)
    pressure = random.uniform(1, 10)

    flame_temp = simulate_flame_temperature(
        phi,
        T_init,
        pressure
    )

    data.append([
        phi,
        T_init,
        pressure,
        flame_temp
    ])

df = pd.DataFrame(
    data,
    columns=[
        "equivalence_ratio",
        "initial_temperature",
        "pressure_atm",
        "flame_temperature"
    ]
)

df.head()

,equivalence_ratio,initial_temperature,pressure_atm,flame_temperature
0,1.046473,476.530994,9.343775,2370.797153
1,0.613199,716.204702,8.966191,2009.369559
2,0.930877,309.146110,5.576190,2196.280125
3,1.353624,329.755551,8.268642,2038.345789
4,0.796630,588.387534,5.907016,2197.079867


In [6]:
from sklearn.model_selection import train_test_split

X = df.drop("flame_temperature", axis=1)
y = df["flame_temperature"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=150),
    "Gradient Boosting": GradientBoostingRegressor(),
    "SVR": SVR(),
    "KNN": KNeighborsRegressor()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append([
        name,
        mean_absolute_error(y_test, preds),
        mean_squared_error(y_test, preds),
        r2_score(y_test, preds)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Model", "MAE", "MSE", "R2 Score"]
)

results_df

,Model,MAE,MSE,R2 Score
0,Linear Regression,105.063193,14725.061382,0.447775
1,Ridge Regression,104.850397,14668.906359,0.449880
2,Lasso Regression,104.583685,14604.128694,0.452310
3,Decision Tree,14.866679,367.498336,0.986218
4,Random Forest,10.232314,185.319714,0.993050
5,Gradient Boosting,10.437932,171.134172,0.993582
6,SVR,117.216085,22699.151369,0.148727
7,KNN,123.781609,23504.762474,0.118514


In [8]:
# Sort by best performance:
# 1) Highest R2 Score
# 2) Lowest MAE
best_model_row = results_df.sort_values(
    by=["R2 Score", "MAE"],
    ascending=[False, True]
).iloc[0]

print("Best Model Selected")
print("-------------------")
print(f"Model Name : {best_model_row['Model']}")
print(f"MAE        : {best_model_row['MAE']:.4f}")
print(f"MSE        : {best_model_row['MSE']:.4f}")
print(f"R2 Score   : {best_model_row['R2 Score']:.6f}")

Best Model Selected
-------------------
Model Name : Gradient Boosting
MAE        : 10.4379
MSE        : 171.1342
R2 Score   : 0.993582
